# NeuralGCM run: Oct 9-12, 2026

This notebook runs **NeuralGCM** and saves daily precipitation.

- **Initial conditions:** ERA5 at 2026-10-09 00Z, downloaded from Google's public archive.
- **Ocean (SST) and sea ice:** ERA5 at the same time, **held constant** for the whole forecast (persistence).
- **Ensemble:** NeuralGCM is stochastic (random). Each **seed** gives a slightly different forecast.

**How to run:** Runtime > Change runtime type > Hardware Accelerator > **T4 GPU**.

Go into Step 2: Settings and change SEEDS to any 5 numbers you pick.

Then click **Run all**. The first seed is slow.



In [ ]:
# =============================================================================
# Step 1: Install
# =============================================================================
!pip install -q neuralgcm gcsfs

In [ ]:
# =============================================================================
# Step 2: Settings
# =============================================================================
INIT_TIME         = "2026-10-09T00"   # start time YYYY-MM-DD{Time}
FORECAST_DAYS     = 4                 # 3 days: Oct 9 00Z -> Oct 13 00Z (covers Oct 9, 10, 11)
OUTPUT_STEP_HOURS = 24                # forecast (output) step: 24 = daily, 6 = 6-hourly, 1 = hourly
SEEDS             = [0, 13, 7, 52, 109]   # pick ANY 5 numbers
OUT_DIR           = "/content/"

In [ ]:
# =============================================================================
# Step 3: LOAD THE MODEL
# =============================================================================
import os, pickle, time
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # must be set before importing jax

import numpy as np
import pandas as pd
import xarray as xr
import jax
import gcsfs
import neuralgcm
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils

print(jax.devices())   # should show a GPU (CudaDevice), not CpuDevice
os.makedirs(OUT_DIR, exist_ok=True)

gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open("gs://neuralgcm/models/v1_precip/stochastic_precip_2_8_deg.pkl", "rb") as f:
    model = neuralgcm.PressureLevelModel.from_checkpoint(pickle.load(f))

In [ ]:
# =============================================================================
# Step 4: Initial Conditions
# =============================================================================

# pull the data from ARCO
init_time = pd.Timestamp(INIT_TIME)
era5 = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                    chunks=None, storage_options={"token": "anon"})
era5_init = era5[model.input_variables + model.forcing_variables].sel(time=[init_time]).load()

# regrid from ERA5's 0.25° grid to the model's 2.8° grid
era5_grid = spherical_harmonic.Grid(
    latitude_nodes=era5_init.sizes["latitude"],
    longitude_nodes=era5_init.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(era5_init.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(era5_init.longitude))
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True)
init_2p8 = xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(era5_init, regridder))

# SST/sea ice
forcings_ds = init_2p8[model.forcing_variables]
print(forcings_ds)

In [ ]:
# =============================================================================
# Step 5: Run NeuralGCM
# =============================================================================
inputs           = model.inputs_from_xarray(init_2p8.isel(time=0))
initial_forcings = model.forcings_from_xarray(forcings_ds.isel(time=0))
forcings         = model.forcings_from_xarray(forcings_ds)
n_steps  = 24 * FORECAST_DAYS // OUTPUT_STEP_HOURS

def keep_precip(state, forcings):
    return model.decode(state, forcings)["precipitation_cumulative_mean"]

def out_path(seed):
    return f"{OUT_DIR}/{init_time:%Y%m%d}_seed{seed}_persist.nc"

#runs all the seeds you pick
for seed in SEEDS:
    if os.path.exists(out_path(seed)):
        print(f"seed {seed}: already done, skipping")
        continue
    t0 = time.time()
    state = model.encode(inputs, initial_forcings, jax.random.PRNGKey(seed))
    _, precip_total = model.unroll(state, forcings, steps=n_steps,
                                   timedelta=np.timedelta64(OUTPUT_STEP_HOURS, "h"),
                                   start_with_input=False,      # outputs at +1 step ... +n_steps
                                   post_process_fn=keep_precip)

    # running total (m) -> precipitation that fell during each step (mm)
    total_mm = np.maximum.accumulate(np.asarray(precip_total).squeeze() * 1000.0, axis=0)
    step_mm  = np.diff(total_mm, axis=0, prepend=0.0)

    valid = pd.date_range(init_time + pd.Timedelta(hours=OUTPUT_STEP_HOURS),
                          periods=n_steps, freq=f"{OUTPUT_STEP_HOURS}h")
    precip = xr.DataArray(step_mm.astype(np.float32), dims=("time", "longitude", "latitude"),
                          coords={"time": valid, "longitude": init_2p8.longitude,
                                  "latitude": init_2p8.latitude},
                          attrs={"units": "mm", "note": "total over the step ending at the time stamp"})
    precip.transpose("time", "latitude", "longitude").to_dataset(name="precipitation").to_netcdf(out_path(seed))
    print(f"seed {seed}: {time.time() - t0:.1f} s  ->  {out_path(seed)}")
print("done")

In [ ]:
# =============================================================================
# Step 6: Download the files in a zip
# =============================================================================
!cd /content && zip -qr ngm_output.zip *.nc
from google.colab import files
files.download("/content/ngm_output.zip")